# Extract Egyptian Modalink Frames (Colab + Google Drive)

**Goal:** Read annotated segments → open videos from Drive → face-crop frames → save labeled images for EfficientNet-B0 fine-tuning.

## Before you run
1. Open this notebook in **Google Colab**
2. **Runtime → Change runtime type → GPU** (optional for extract; useful later for fine-tune)
3. In Google Drive (browser):
   - Open [Final Modalink Dataset](https://drive.google.com/drive/folders/1ZXR5n4ry5RCdhPvOmb_ugf4sYc9FwdLH)
   - Right-click → **Organize → Add shortcut to Drive** → put it in **My Drive** (e.g. `My Drive/Final Modalink Dataset`)
   - Confirm your frames folder exists: [Data](https://drive.google.com/drive/folders/1grfrZ9ALe8kdoFRPCaliT4mSapbApZVz)
     (also add a shortcut under My Drive if it is only under Shared with me)
4. Upload `annotations version 2.xlsx` in the cell below **or** place it on Drive and set `ANNOTATIONS_XLSX`

## Label rule
- Use **`Final Overall (majority of modalities)`**
- Map Happiness→happy, Sadness→sad, …
- Skip Ambiguous / empty
- Person key for later split: `Folder|speaker`

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
print("Mounted. Top-level My Drive entries:")
for p in sorted(Path("/content/drive/MyDrive").iterdir())[:30]:
    print(" -", p.name)

In [ ]:
# 2) Paths — EDIT if your shortcut names differ
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

# Source videos (shortcut name after "Add shortcut to Drive")
MODALINK_ROOT = MYDRIVE / "Final Modalink Dataset"

# Where to store extracted face frames
# Option A: shortcut named "Data" in My Drive
FRAMES_ROOT = MYDRIVE / "Data" / "egypt_modalink_frames"
# Option B: if Data shortcut is nested, fix path manually, e.g.:
# FRAMES_ROOT = MYDRIVE / "Master Documents and ..." / "Data" / "egypt_modalink_frames"

# Annotations: upload in next cell OR point to a Drive copy
ANNOTATIONS_XLSX = Path("/content/annotations_version_2.xlsx")

FRAMES_ROOT.mkdir(parents=True, exist_ok=True)
print("MODALINK_ROOT exists:", MODALINK_ROOT.exists(), "→", MODALINK_ROOT)
print("FRAMES_ROOT:", FRAMES_ROOT)

if not MODALINK_ROOT.exists():
    print("\nERROR: Modalink root not found.")
    print("Add a shortcut of 'Final Modalink Dataset' into My Drive, then re-run.")
    print("Candidates containing 'Modalink' or 'videoplayback':")
    for p in MYDRIVE.rglob("*"):
        if p.is_dir() and ("modalink" in p.name.lower() or p.name.startswith("videoplayback")):
            print(" ", p)
            if p.name.startswith("videoplayback"):
                break

In [ ]:
# 3) Provide annotations Excel (upload if not already on Drive)
from google.colab import files
import shutil

if not ANNOTATIONS_XLSX.exists():
    # Try common Drive locations first
    candidates = list(MYDRIVE.rglob("*annotations*version*2*.xlsx"))
    candidates += list(MYDRIVE.rglob("*annotations version 2.xlsx"))
    if candidates:
        ANNOTATIONS_XLSX = candidates[0]
        print("Found on Drive:", ANNOTATIONS_XLSX)
    else:
        print("Upload annotations version 2.xlsx ...")
        uploaded = files.upload()
        name = next(iter(uploaded))
        ANNOTATIONS_XLSX = Path("/content") / name
        print("Uploaded:", ANNOTATIONS_XLSX)
else:
    print("Using:", ANNOTATIONS_XLSX)

In [ ]:
# 4) Config + label mapping
import re
import cv2
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

LABEL_COL = "Final Overall (majority of modalities)"

EMOTION_MAP = {
    "Anger": "angry",
    "Disgust": "disgust",
    "Fear": "fear",
    "Happiness": "happy",
    "Neutral": "neutral",
    "Sadness": "sad",
    "Surprise": "surprise",
}
SKIP_LABELS = {"", "nan", "none", "ambiguous", "amiguous"}

FPS_SAMPLE = 2.0          # ~2 frames per second
MAX_FACES_PER_SEGMENT = 15
MIN_FACE_SIZE = 60
FACE_PAD = 0.25
SAVE_SIZE = 96            # matches your EfficientNet .h5 input
DRY_RUN_LIMIT = None      # e.g. 20 to test first; None = all rows

CASCADE = cv2.CascadeClassifier(
    str(Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml")
)
assert not CASCADE.empty()

df = pd.read_excel(ANNOTATIONS_XLSX)
print("Rows:", len(df))
print(df[LABEL_COL].value_counts(dropna=False))

In [ ]:
# 5) Build worklist from Excel

def normalize_label(raw) -> str | None:
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    if s.lower() in SKIP_LABELS:
        return None
    return EMOTION_MAP.get(s)

def resolve_video_path(folder: str, video_file: str) -> Path | None:
    """Excel video_file looks like: videos/SPEAKER_00/SPEAKER_00_segment_0000.mp4"""
    rel = Path(str(folder)) / str(video_file)
    full = MODALINK_ROOT / rel
    if full.exists():
        return full
    # fallback: search by filename inside that Folder
    folder_dir = MODALINK_ROOT / str(folder)
    name = Path(str(video_file)).name
    if folder_dir.exists():
        hits = list(folder_dir.rglob(name))
        if hits:
            return hits[0]
    return None

rows = []
skipped = {"no_label": 0, "missing_video": 0}

for i, r in df.iterrows():
    emotion = normalize_label(r.get(LABEL_COL))
    if emotion is None:
        skipped["no_label"] += 1
        continue
    folder = str(r["Folder"]).strip()
    video_file = str(r["video_file"]).strip()
    speaker = str(r["speaker"]).strip()
    person_id = f"{folder}|{speaker}"
    vpath = resolve_video_path(folder, video_file)
    if vpath is None:
        skipped["missing_video"] += 1
        continue
    rows.append({
        "excel_row": int(i),
        "folder": folder,
        "speaker": speaker,
        "person_id": person_id,
        "emotion": emotion,
        "video_path": str(vpath),
        "segment_id": r.get("segment_id"),
        "speaker_segment_id": r.get("speaker_segment_id"),
    })

work = pd.DataFrame(rows)
if DRY_RUN_LIMIT:
    work = work.head(DRY_RUN_LIMIT)

print("Usable segments:", len(work))
print("Skipped:", skipped)
print("Emotion counts:\n", work["emotion"].value_counts())
print("Unique persons (Folder|speaker):", work["person_id"].nunique())
work.head(3)

In [ ]:
# 6) Face crop helper + extract one segment

def largest_face_bgr(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = CASCADE.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(MIN_FACE_SIZE, MIN_FACE_SIZE)
    )
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    pad_x, pad_y = int(w * FACE_PAD), int(h * FACE_PAD)
    x1, y1 = max(0, x - pad_x), max(0, y - pad_y)
    x2 = min(frame_bgr.shape[1], x + w + pad_x)
    y2 = min(frame_bgr.shape[0], y + h + pad_y)
    return frame_bgr[y1:y2, x1:x2]


def safe_stem(person_id: str, emotion: str, segment_id) -> str:
    pid = re.sub(r"[^A-Za-z0-9_|-]+", "_", person_id)
    return f"{pid}__{emotion}__seg{segment_id}"


def extract_faces_from_video(video_path: Path, out_dir: Path, stem: str) -> list[str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    step = max(1, int(round(fps / FPS_SAMPLE)))
    saved = []
    idx = 0
    kept = 0

    while kept < MAX_FACES_PER_SEGMENT:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            face = largest_face_bgr(frame)
            if face is not None and face.size > 0:
                face = cv2.resize(face, (SAVE_SIZE, SAVE_SIZE), interpolation=cv2.INTER_AREA)
                out_path = out_dir / f"{stem}__f{kept:03d}.jpg"
                cv2.imwrite(str(out_path), face, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                saved.append(str(out_path))
                kept += 1
        idx += 1

    cap.release()
    return saved

In [ ]:
# 7) Run extraction (writes into Drive FRAMES_ROOT)
import json

manifest_rows = []
n_images = 0
n_empty = 0

for _, row in tqdm(work.iterrows(), total=len(work)):
    emotion = row["emotion"]
    person_id = row["person_id"]
    out_dir = FRAMES_ROOT / "by_emotion" / emotion / person_id.replace("|", "__")
    stem = safe_stem(person_id, emotion, row["speaker_segment_id"])
    paths = extract_faces_from_video(Path(row["video_path"]), out_dir, stem)
    if not paths:
        n_empty += 1
        continue
    for p in paths:
        manifest_rows.append({
            "image_path": p,
            "emotion": emotion,
            "person_id": person_id,
            "folder": row["folder"],
            "speaker": row["speaker"],
            "video_path": row["video_path"],
            "excel_row": row["excel_row"],
        })
        n_images += 1

manifest = pd.DataFrame(manifest_rows)
manifest_path = FRAMES_ROOT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

summary = {
    "n_segments_attempted": int(len(work)),
    "n_segments_with_faces": int(manifest["video_path"].nunique()) if len(manifest) else 0,
    "n_segments_no_face": int(n_empty),
    "n_images": int(n_images),
    "emotion_counts": manifest["emotion"].value_counts().to_dict() if len(manifest) else {},
    "n_persons": int(manifest["person_id"].nunique()) if len(manifest) else 0,
    "label_column": LABEL_COL,
    "frames_root": str(FRAMES_ROOT),
}
with open(FRAMES_ROOT / "extract_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Manifest:", manifest_path)

## Output layout on Drive

```text
Data/egypt_modalink_frames/
  by_emotion/
    angry/{person_id}/*.jpg
    disgust/...
    fear/...
    happy/...
    neutral/...
    sad/...
    surprise/...
  manifest.csv
  extract_summary.json
```

## Next step (after extraction)
Fine-tune `best_model.h5` on these frames with:
- person-level split (`person_id`)
- class weights (imbalance)

Tip: set `DRY_RUN_LIMIT = 20` first to verify paths, then set `None` and re-run.